# LAB 01 — Spark Runtime & Spark UI
**Môn:** Phân tích Dữ liệu Lớn với Spark  
**Phần kiến thức áp dụng:** (1) Kiến trúc Runtime & Giám sát · (2) Spark UI: Đọc hiểu & Phát hiện bottleneck

## Mục tiêu
Sau khi hoàn thành lab, sinh viên có thể:
1. Phân biệt được **Job → Stage → Task** trong một pipeline Spark thực tế.
2. Quan sát và giải thích các thông số ở **Spark UI** (DAG view, Stage view, Executor tab).
3. Phát hiện các vấn đề: **data skew, GC pressure, OOM, shuffle nặng**.
4. Đề xuất giải pháp tối ưu (repartition, salting, broadcast join, cache).

## Yêu cầu môi trường
- Python 3.9+, PySpark 3.4+ (`pip install pyspark`)
- Java 8/11/17
- Trình duyệt để mở Spark UI (mặc định http://localhost:4040)

> Sau mỗi action (`count`, `show`, `collect`, `write`), hãy mở Spark UI và quan sát Job vừa chạy **TRƯỚC KHI** chạy cell tiếp theo.

---
## Phần 0 · Khởi tạo SparkSession và sinh dữ liệu
Chạy cell này trước. Lab dùng dữ liệu **giao dịch bán hàng giả lập** với data skew có chủ ý.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random, time

spark = (SparkSession.builder
    .appName("Lab01-Spark-Runtime-UI")
    .master("local[4]")                    # 4 core cho 4 executor slot
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # TẮT AQE để quan sát skew
    .config("spark.ui.port", "4040")
    .getOrCreate())

sc = spark.sparkContext
print("Spark version:", spark.version)
print("Spark UI:", sc.uiWebUrl)            # ← Mở link này trên trình duyệt
print("Default parallelism:", sc.defaultParallelism)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 13:54:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.8
Spark UI: http://notebook:4040
Default parallelism: 4


In [2]:
from pyspark.sql import SparkSession

def get_spark(mode="local"):
    """mode: 'local' hoặc 'cluster'"""
    builder = SparkSession.builder.appName(f"Lab-{mode}")
    
    if mode == "local":
        builder = builder.master("local[4]")
    elif mode == "cluster":
        builder = (builder
            .master("spark://master:7077")
            .config("spark.driver.host", "notebook")
            .config("spark.driver.bindAddress", "0.0.0.0")
            .config("spark.driver.port", "7001")
            .config("spark.blockManager.port", "7002")
            .config("spark.executor.memory", "2g")
            .config("spark.executor.cores", "2")
            .config("spark.cores.max", "4"))
    
    return (builder
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate())

def get_data_dir(mode="local"):
    return ("file:///opt/workspace/data" if mode == "local"
            else "hdfs://master:9000/user/root/data")

# === Sử dụng ===
MODE = "cluster"   # đổi giữa "local" và "cluster"
spark = get_spark(MODE)
DATA_DIR = get_data_dir(MODE)

print(f"Chế độ: {MODE}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"Data dir: {DATA_DIR}")

Chế độ: cluster
Spark UI: http://notebook:4040
Data dir: hdfs://master:9000/user/root/data


26/05/08 13:54:25 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
# Sinh 5 triệu giao dịch — 80% rơi vào product_id = 1 (data skew)
N_ROWS = 1000

def gen_partition(idx, iterator):
    rnd = random.Random(42 + idx)
    for _ in iterator:
        # 80% xác suất là product_id=1 (hot key)
        pid = 1 if rnd.random() < 0.8 else rnd.randint(2, 1000)
        yield (
            rnd.randint(1, 100000),                 # user_id
            pid,                                      # product_id (SKEWED)
            round(rnd.uniform(10, 500), 2),           # amount
            f"2025-{rnd.randint(1,12):02d}-{rnd.randint(1,28):02d}"  # date
        )

rdd = sc.parallelize(range(N_ROWS), numSlices=8).mapPartitionsWithIndex(gen_partition)
schema = StructType([
    StructField("user_id",    IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("amount",     DoubleType()),
    StructField("date",       StringType()),
])

transactions = spark.createDataFrame(rdd, schema)
transactions.write.mode("overwrite").parquet("file:///opt/workspace/data/module4_transactions")

# Bảng products (nhỏ, dùng cho join)
products = spark.range(1, 1001).select(
    F.col("id").alias("product_id"),
    F.concat(F.lit("Product_"), F.col("id")).alias("name"),
    (F.col("id") % 10).alias("category_id")
)
products.write.mode("overwrite").parquet("file:///opt/workspace/data/module4_products")
print("✅ Đã tạo dữ liệu tại data/module4_transactions và data/module4_products")

✅ Đã tạo dữ liệu tại data/module4_transactions và data/module4_products


In [2]:
path_transaction_data = "file:///opt/workspace/data/module4_transactions"
path_product_data = "file:///opt/workspace/data/module4_products"

---
## Bài 1 · Job → Stage → Task
**Mục tiêu:** Quan sát cách Spark chia 1 query thành Jobs, mỗi Job thành Stages (cắt bởi shuffle), mỗi Stage thành Tasks (= số partition).

Chạy cell dưới rồi vào **Spark UI → Jobs tab**, ghi lại số liệu vào **Bảng 1** trong báo cáo.

In [3]:
tx = spark.read.parquet(path_transaction_data)
pr = spark.read.parquet(path_product_data)

# Query A: chỉ filter + count → có shuffle không?
print("--- Query A ---")
t0 = time.time()
n = tx.filter(F.col("amount") > 100).count()
print(f"Count = {n}, time = {time.time()-t0:.2f}s")

--- Query A ---


Count = 824, time = 6.86s


In [4]:
# Query B: groupBy + sum → có shuffle
print("--- Query B ---")
t0 = time.time()
agg = tx.groupBy("product_id").agg(F.sum("amount").alias("total"))
agg.count()  # action thứ 2
print(f"time = {time.time()-t0:.2f}s")

# Hiển thị physical plan
agg.explain(mode="formatted")

--- Query B ---


time = 10.26s
== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * ColumnarToRow (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [product_id#1, amount#2]
Batched: true
Location: InMemoryFileIndex [file:/opt/workspace/data/module4_transactions]
ReadSchema: struct<product_id:int,amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [product_id#1, amount#2]

(3) HashAggregate [codegen id : 1]
Input [2]: [product_id#1, amount#2]
Keys [1]: [product_id#1]
Functions [1]: [partial_sum(amount#2)]
Aggregate Attributes [1]: [sum#39]
Results [2]: [product_id#1, sum#40]

(4) Exchange
Input [2]: [product_id#1, sum#40]
Arguments: hashpartitioning(product_id#1, 8), ENSURE_REQUIREMENTS, [plan_id=120]

(5) HashAggregate [codegen id : 2]
Input [2]: [product_id#1, sum#40]
Keys [1]: [product_id#1]
Functions [1]: [sum(amount#2)]
Aggregate Attributes [1]: [sum(amount#2)#28]
Results [2]: [product_id#1, sum(amount#2)#28 AS total#29]




In [5]:
# Query C: join + groupBy → nhiều shuffle hơn
print("--- Query C ---")
t0 = time.time()
joined = tx.join(pr, "product_id").groupBy("category_id").agg(F.sum("amount").alias("total"))
joined.show()
print(f"time = {time.time()-t0:.2f}s")

--- Query C ---


+-----------+------------------+
|category_id|             total|
+-----------+------------------+
|          2|           3851.16|
|          9|           4531.92|
|          3|           5957.92|
|          7| 6424.719999999999|
|          5| 6002.129999999998|
|          4|            4690.2|
|          8| 4924.910000000001|
|          1|210549.75000000003|
|          6|           4815.83|
|          0| 7125.219999999999|
+-----------+------------------+

time = 16.13s


### 📝 Câu hỏi 1
Điền **Bảng 1** trong báo cáo (Query A/B/C → số Job, số Stage, số Task của stage cuối, có shuffle hay không).  
Vì sao Query A có ít stage hơn Query B?

---
## Bài 2 · Đọc hiểu DAG View
Vào **SQL / DataFrame tab** → click query gần nhất → xem DAG.  
Trả lời:
- Có bao nhiêu node `Exchange` (= shuffle) trong Query C?
- `HashAggregate` xuất hiện mấy lần? Vì sao có **partial aggregate** trước shuffle và **final aggregate** sau shuffle?

In [7]:
# Chạy thêm 1 query với window function để DAG phức tạp hơn
from pyspark.sql.window import Window

w = Window.partitionBy("product_id").orderBy(F.desc("amount"))
top_per_product = (tx.withColumn("rn", F.row_number().over(w))
                     .filter("rn <= 3")
                     .groupBy("product_id").agg(F.avg("amount")))
top_per_product.count()
top_per_product.explain(mode="formatted")

== Physical Plan ==
* HashAggregate (12)
+- * HashAggregate (11)
   +- * Project (10)
      +- * Filter (9)
         +- Window (8)
            +- WindowGroupLimit (7)
               +- * Sort (6)
                  +- Exchange (5)
                     +- WindowGroupLimit (4)
                        +- * Sort (3)
                           +- * ColumnarToRow (2)
                              +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [product_id#1, amount#2]
Batched: true
Location: InMemoryFileIndex [file:/opt/workspace/data/module4_transactions]
ReadSchema: struct<product_id:int,amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [product_id#1, amount#2]

(3) Sort [codegen id : 1]
Input [2]: [product_id#1, amount#2]
Arguments: [product_id#1 ASC NULLS FIRST, amount#2 DESC NULLS LAST], false, 0

(4) WindowGroupLimit
Input [2]: [product_id#1, amount#2]
Arguments: [product_id#1], [amount#2 DESC NULLS LAST], row_number(), 3, Partial

(5) Exchange
Input [2]: [product_id#1,

---
## Bài 3 · Phát hiện DATA SKEW
Vào **Stages tab** → click stage chứa shuffle write của Query B → xem cột **Summary Metrics for Tasks**.  
So sánh **Min / Median / Max** của *Shuffle Read Size* và *Duration*.

**Dấu hiệu skew:** Max ≫ Median (gấp 5×, 10× hoặc hơn).

In [4]:
# Thống kê phân bố key — confirm skew
tx.groupBy("product_id").count().orderBy(F.desc("count")).show(10)

+----------+-----+
|product_id|count|
+----------+-----+
|         1|  787|
|       426|    3|
|       670|    3|
|       317|    3|
|       343|    3|
|       301|    2|
|       553|    2|
|       524|    2|
|       958|    2|
|       850|    2|
+----------+-----+
only showing top 10 rows



In [ ]:
# Giải pháp 1: SALTING — tách hot key thành nhiều sub-key
SALT_BUCKETS = 16
tx_salted = tx.withColumn("salt", (F.rand()*SALT_BUCKETS).cast("int"))

t0 = time.time()
agg_salted = (tx_salted
    .groupBy("product_id", "salt").agg(F.sum("amount").alias("part_total"))
    .groupBy("product_id").agg(F.sum("part_total").alias("total")))
agg_salted.count()
print(f"Salted version: {time.time()-t0:.2f}s")

In [ ]:
# Giải pháp 2: BẬT AQE (Adaptive Query Execution) → tự động xử lý skew
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

t0 = time.time()
tx.groupBy("product_id").agg(F.sum("amount")).count()
print(f"AQE version: {time.time()-t0:.2f}s")

# TẮT lại để các bài sau vẫn quan sát được skew
spark.conf.set("spark.sql.adaptive.enabled", "false")

### 📝 Câu hỏi 3
- Ghi lại Min / Median / Max thời gian task của Query B (trước salting).
- So sánh thời gian Query B gốc vs salted vs AQE.
- Nếu hot key chiếm 99% dữ liệu thay vì 80%, bạn còn dùng salting không? Vì sao?

---
## Bài 4 · Executor tab — GC, Memory, Shuffle
Mở **Executors tab**. Quan sát các cột:
- *Task Time (GC Time)* — nếu GC > 10% Task Time là **GC pressure**.
- *Shuffle Read / Write* — đo I/O shuffle.
- *Storage Memory* — bộ nhớ dùng cho cache.

Chạy cell dưới để tạo áp lực memory:

In [ ]:
# Cache một DataFrame lớn — quan sát Storage Memory tăng
cached = tx.filter(F.col("amount") > 50).cache()
print("Count 1:", cached.count())   # action đầu tiên → trigger cache
print("Count 2:", cached.count())   # action thứ 2 → đọc từ cache (rất nhanh)

# Vào Storage tab xem entry vừa cache

In [ ]:
cached.unpersist()  # giải phóng memory

---
## Bài 5 · Tối ưu JOIN — Broadcast vs Shuffle Hash
Bảng `products` chỉ có 1000 dòng → ứng viên broadcast hoàn hảo.

In [ ]:
# Trước: shuffle join (sort-merge) — 2 shuffle
t0 = time.time()
tx.join(pr, "product_id").count()
print(f"Shuffle join: {time.time()-t0:.2f}s")

In [ ]:
# Sau: broadcast join — 0 shuffle ở phía tx
t0 = time.time()
tx.join(F.broadcast(pr), "product_id").count()
print(f"Broadcast join: {time.time()-t0:.2f}s")

tx.join(F.broadcast(pr), "product_id").explain(mode="formatted")

### 📝 Câu hỏi 5
- Trong physical plan, node nào thay đổi từ `SortMergeJoin` sang `BroadcastHashJoin`?
- Khi nào KHÔNG nên dùng broadcast? (gợi ý: ngưỡng `spark.sql.autoBroadcastJoinThreshold`, mặc định 10MB)

---
## Bài 6 (★ Thử thách) · Tái hiện và sửa OOM
Cell dưới **CỐ Ý** gây OOM hoặc rất chậm bằng `collect()` toàn bộ data về driver.

In [ ]:
# ⚠️ CHẠY THỬ — sẽ chậm hoặc lỗi OOM nếu dữ liệu lớn
try:
    bad = tx.collect()       # KÉO 5 TRIỆU DÒNG VỀ DRIVER
    print("len:", len(bad))
except Exception as e:
    print("Lỗi:", type(e).__name__, str(e)[:200])

In [ ]:
# Cách sửa đúng: aggregate trước, hoặc dùng take/limit
good = tx.groupBy("product_id").agg(F.sum("amount").alias("total")).collect()
print("Số dòng kéo về driver:", len(good))   # chỉ ~1000 dòng

In [6]:
spark.stop()

### 📝 Câu hỏi 6
- Lỗi OOM xảy ra ở **driver** hay **executor**? Vì sao?
- Liệt kê 3 thói quen lập trình PySpark dễ gây OOM driver.

---
## Kết thúc
Trước khi nộp:
1. Chụp ảnh **Jobs tab**, **DAG của Query C**, **Stage có skew (Summary Metrics)**, **Executors tab**.
2. Hoàn thành báo cáo `BaoCao_Lab01.docx` (file đính kèm theo lab).
3. Nộp: `notebook.ipynb` + `BaoCao_Lab01.docx` (đã chèn screenshot).

In [9]:
spark.stop()

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
/opt/spark/python/lib/pyspark.zip/pyspark/context.py:657: RuntimeWarning: Unable to cleanly shutdown Spark JVM process. It is possible that the process has crashed, been killed or may also be in a zo

ConnectionRefusedError: [Errno 111] Connection refused